In [3]:
import json
import os
from datetime import date
from functools import partial
from typing import Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import porepy as pp

from src.tpf_lab.ml.rel_perm import brookscorey_w, error_func_deriv
from src.tpf_lab.models.run_models import run_time_dependent_model
from src.tpf_lab.models.two_phase_flow import TwoPhaseFlow
from src.tpf_lab.numerics.ad.functions import pow
from src.tpf_lab.utils import rm_out_padding

rm_out_padding()

AttributeError: module 'porepy.models' has no attribute 'abstract_model'

### Setup
For the problem setup, look into `two_phase_flow_setup_runs.ipynb`.

In [2]:
w_source_cell_index = 209

class ModifiedModel(TwoPhaseFlow):
    def _w_source(self, g: pp.Grid) -> np.ndarray:
        array: np.ndarray = super()._w_source(g)
        array[w_source_cell_index] = 0.2
        return array

#### Sanity runs for ``two_phase_flow_artificial_error_runs.ipynb``
To check that the results of the artificial error runs are reliable, we make the same
tests with the unmodified rel. perm. model.

In [ ]:
model = ModifiedModel(
    {
        "file_name": f"{date.today().strftime('%Y-%m-%d')}",
        "folder_name": os.path.join(
            "results",
            "artificial_error_sanity_tests",
            f"pcapmodel_None_relpermmodel_Brooks-Corey_wsourcecellindex_{w_source_cell_index}"
            ),
    }
)
model._formulation = "n_pressure_w_saturation"
model._cap_pressure_model = "None"
model._schedule: np.ndarray = np.array([0, 1.0])
run_time_dependent_model(model, {"max_iterations": 30})

In [ ]:
model = ModifiedModel(
    {
        "file_name": f"{date.today().strftime('%Y-%m-%d')}",
        "folder_name": os.path.join(
            "results",
            "artificial_error_sanity_tests",
            f"pcapmodel_linear_relpermmodel_Brooks-Corey_wsourcecellindex_{w_source_cell_index}"
            ),
    }
)
model._formulation = "n_pressure_w_saturation"
model._cap_pressure_model = "linear"
model._schedule: np.ndarray = np.array([0, 1.0])
run_time_dependent_model(model, {"max_iterations": 30})

In [ ]:
model = ModifiedModel(
    {
        "file_name": f"{date.today().strftime('%Y-%m-%d')}",
        "folder_name": os.path.join(
            "results",
            "artificial_error_sanity_tests",
            "artifical_error_relperm_w",
            f"pcapmodel_van_Genuchten_relpermmodel_Brooks-Corey_wsourcecellindex_{w_source_cell_index}"
            ),
    }
)
model._formulation = "n_pressure_w_saturation"
model._cap_pressure_model = "van Genuchten"
model._schedule: np.ndarray = np.array([0, 1.0])
run_time_dependent_model(model, {"max_iterations": 30})

In [7]:
model = ModifiedModel(
    {
        "file_name": f"{date.today().strftime('%Y-%m-%d')}",
        "folder_name": os.path.join(
            "results",
            "artificial_error_tests",
            "artifical_error_relperm_w",
            f"pcapmodel_Brooks-Corey_relpermmodel_Brooks-Corey_wsourcecellindex_{w_source_cell_index}"
            ),
    }
)
model._formulation = "n_pressure_w_saturation"
model._cap_pressure_model = "Brooks-Corey"
model._schedule: np.ndarray = np.array([0, 1.0])
run_time_dependent_model(model, {"max_iterations": 30})

time loop:   0%|          | 0/6 [00:00<?, ?it/s]

Newton loop:   0%|          | 0/30 [00:00<?, ?it/s]

Newton loop:   0%|          | 0/30 [00:00<?, ?it/s]

Newton loop:   0%|          | 0/30 [00:00<?, ?it/s]

Newton loop:   0%|          | 0/30 [00:00<?, ?it/s]

Newton loop:   0%|          | 0/30 [00:00<?, ?it/s]

#### Varying the position of the source
We observed by accident that the stability is very dependent on the position of the
wetting source. Thus we test out different positions, i.e.
$$
f_{w,i}=\begin{cases}
0.2,\quad&\text{if }i=I,\\
0,&\text{otherwise},
\end{cases}
$$
where $V_{I}=\{(x,y)\mid 0.9\leq x\leq1, k\cdot0.2\leq y\leq k\cdot0.2+0.1\}$ and we
vary $k.$

In [ ]:
class VaryingWSourceModel(TwoPhaseFlow):

    def __init__(self, params: Optional[dict] = None) -> None:
        super().__init__(params)
        self._w_source_x_index: int = 10
        self._w_source_y_index: int = 10

    def _w_source(self, g: pp.Grid) -> np.ndarray:
        cell_index = self._w_source_y_index * 20 + self._w_source_x_index
        array: np.ndarray = super()._w_source(g)
        array[cell_index] = 0.2
        return array

In [ ]:
cap_pressure_model = "Brooks-Corey"

failure_time_indices = []

for x_index in range(0, 20, 1):
    failure_time_indices_y = []
    for y_index in range(0, 20, 1):
        model = VaryingWSourceModel(
            {
                "file_name": f"{date.today().strftime('%Y-%m-%d')}",
                "folder_name": os.path.join(
                    "results",
                    "artificial_error_sanity_tests",
                    "varying_w_source_position",
                    f"pcapmodel_{cap_pressure_model}_relpermmodel_Brooks-Corey",
                    f"y_source_index_{y_index}_x_source_index_{x_index}"
                    ),
            }
        )
        model._formulation = "n_pressure_w_saturation"
        model._cap_pressure_model = cap_pressure_model
        model._w_source_x_index = x_index
        model._w_source_y_index = y_index
        model._schedule: np.ndarray = np.array([0, 1.0])
        try:
            run_time_dependent_model(model, {"max_iterations": 30})
        except Exception:
            pass
        failure_time_indices_y.append(model.time_manager.time_index)
    failure_time_indices.append(failure_time_indices_y)

with open(os.path.join(
    "results",
    "artificial_error_sanity_tests",
    "varying_w_source_position",
    f"pcapmodel_{cap_pressure_model}_relpermmodel_Brooks-Corey",
    "failure_timesteps.txt"), "w") as f:
    json.dump(failure_time_indices, f)
plt.imshow(np.array(failure_time_indices), extent=(0, 2, 0, 2), origin="lower")
plt.xlabel("x")
plt.ylabel("y")
plt.colorbar()
plt.savefig(os.path.join(
    "results",
    "artificial_error_sanity_tests",
    "varying_w_source_position",
    f"pcapmodel_{cap_pressure_model}_relpermmodel_Brooks-Corey",
    "failure_timesteps.png"))

#### Varying the offset
As the initial wetting saturation is $S_w=0.5$ for all grid cells, it is in the region of the
rel. perm. curve with the artificial error. To test out the influence of the error (and
also the monotonically increasing part of the error vs. the monotonically decreasing
part of the error), we vary the offset $offset\in\{0.4, 0.6\}.$

In [ ]:
cap_pressure_model = "Brooks-Corey"

failure_time_indices = []

offsets = np.arange(0.4, 0.6, 0.005)

for offset in offsets:
    model = ModifiedModel(
            {
                "file_name": f"{date.today().strftime('%Y-%m-%d')}",
                "folder_name": os.path.join(
                    "results",
                    "artificial_error_sanity_tests",
                    "varying_offset",
                    f"pcapmodel_{cap_pressure_model}_relpermmodel_Brooks-Corey",
                    f"offset_{offset}"
                    ),
            }
        )

    model._formulation = "n_pressure_w_saturation"
    model._cap_pressure_model = "cap_pressure_model"
    model._offset = offset
    model._schedule: np.ndarray = np.array([0, 10.0])
    try:
        run_time_dependent_model(model, {"max_iterations": 30})
    except Exception:
        pass
    failure_time_indices.append(model.time_manager.time_index)

plt.scatter(offsets, np.array(failure_time_indices))
plt.xlabel("offset")
plt.ylabel("failure time index (50=no failure)")
plt.savefig(os.path.join(
    "results",
    "artificial_error_sanity_tests",
    "varying_offset",
    f"pcapmodel_{cap_pressure_model}_relpermmodel_Brooks-Corey",
    "failure_timesteps.png"))